# Option Pricing Tutorial

This notebook demonstrates how to use the `myOptionPricing` module for:
- Pricing call and put options using Black-Scholes model
- Calculating option Greeks (Delta, Gamma, Vega)
- Computing implied volatility from market prices
- Creating synthetic option chains

## Installation Requirements
```
pip install numpy pandas scipy matplotlib
```

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from myPortfolioManagement.myOptionPricing import (
    black_scholes_call,
    black_scholes_put,
    option_delta,
    option_gamma,
    option_vega,
    implied_volatility,
    create_option_chain
)

# Set display options
pd.set_option('display.precision', 4)
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Basic Option Pricing

Let's start with a simple example of pricing European options.
We'll use a stock trading at $100 with a strike price of $100 (at-the-money).

In [ ]:
# Basic example parameters
S = 100      # Current stock price
K = 100      # Strike price
T = 1.0      # Time to expiration (1 year)
r = 0.05     # Risk-free rate (5%)
sigma = 0.20 # Volatility (20%)

# Calculate option prices
call_price = black_scholes_call(S, K, T, r, sigma)
put_price = black_scholes_put(S, K, T, r, sigma)

print(f"Stock Price: ${S}")
print(f"Strike Price: ${K}")
print(f"Time to Expiration: {T} years")
print(f"Risk-Free Rate: {r*100}%")
print(f"Volatility: {sigma*100}%")
print(f"\nCall Option Price: ${call_price:.2f}")
print(f"Put Option Price: ${put_price:.2f}")

# Verify put-call parity
parity_check = call_price - put_price - (S - K * np.exp(-r * T))
print(f"\nPut-Call Parity Check: {abs(parity_check):.6f} (should be ~0)")

## 2. Pricing Options Across Different Strike Prices

Let's see how option prices change with different strike prices.

In [ ]:
# Price vs Strike
strikes = np.arange(80, 121, 2)
call_prices = [black_scholes_call(S, K, T, r, sigma) for K in strikes]
put_prices = [black_scholes_put(S, K, T, r, sigma) for K in strikes]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(strikes, call_prices, 'b-', linewidth=2, label='Call Price')
plt.axvline(x=S, color='r', linestyle='--', alpha=0.5, label=f'Current Price ${S}')
plt.xlabel('Strike Price ($)')
plt.ylabel('Option Price ($)')
plt.title('Call Option Price vs Strike Price')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(strikes, put_prices, 'g-', linewidth=2, label='Put Price')
plt.axvline(x=S, color='r', linestyle='--', alpha=0.5, label=f'Current Price ${S}')
plt.xlabel('Strike Price ($)')
plt.ylabel('Option Price ($)')
plt.title('Put Option Price vs Strike Price')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 3. Pricing Options Across Different Time to Expiration

Time decay (theta) is crucial for option traders.

In [ ]:
# Price vs Time
times = np.linspace(0.01, 2, 50)
K = 100  # At-the-money

call_prices_time = [black_scholes_call(S, K, t, r, sigma) for t in times]
put_prices_time = [black_scholes_put(S, K, t, r, sigma) for t in times]

plt.figure(figsize=(10, 5))
plt.plot(times, call_prices_time, 'b-', linewidth=2, label='Call Price')
plt.plot(times, put_prices_time, 'g-', linewidth=2, label='Put Price')
plt.xlabel('Time to Expiration (Years)')
plt.ylabel('Option Price ($)')
plt.title('Option Price vs Time to Expiration (ATM Options)')
plt.legend()
plt.grid(True)
plt.show()

## 4. Option Greeks

Greeks measure the sensitivity of option prices to various factors.

In [ ]:
# Calculate all Greeks
S = 100
K = 100
T = 1.0
r = 0.05
sigma = 0.20

delta_call = option_delta(S, K, T, r, sigma, 'call')
delta_put = option_delta(S, K, T, r, sigma, 'put')
gamma = option_gamma(S, K, T, r, sigma)
vega = option_vega(S, K, T, r, sigma)

print("Option Greeks (ATM Option):")
print(f"{'Greek':<15} {'Call':>10} {'Put':>10}")
print("-" * 35)
print(f"{'Delta':<15} {delta_call:>10.4f} {delta_put:>10.4f}")
print(f"{'Gamma':<15} {gamma:>10.4f} {gamma:>10.4f}")
print(f"{'Vega':<15} {vega:>10.4f} {vega:>10.4f}")

### Delta: Sensitivity to Stock Price

Delta shows how much the option price changes for a $1 change in stock price.

In [ ]:
# Delta visualization
strikes_greek = np.arange(70, 131, 1)
deltas_call = [option_delta(S, K, T, r, sigma, 'call') for K in strikes_greek]
deltas_put = [option_delta(S, K, T, r, sigma, 'put') for K in strikes_greek]

plt.figure(figsize=(10, 5))
plt.plot(strikes_greek, deltas_call, 'b-', linewidth=2, label='Call Delta')
plt.plot(strikes_greek, deltas_put, 'g-', linewidth=2, label='Put Delta')
plt.axvline(x=S, color='r', linestyle='--', alpha=0.5, label=f'Current Price ${S}')
plt.axhline(y=0, color='k', linestyle='-', alpha=0.3)
plt.xlabel('Strike Price ($)')
plt.ylabel('Delta')
plt.title('Option Delta vs Strike Price')
plt.legend()
plt.grid(True)
plt.show()

### Gamma: Sensitivity of Delta

Gamma peaks at-the-money and decreases as options move in or out of the money.

In [ ]:
# Gamma visualization
gammas = [option_gamma(S, K, T, r, sigma) for K in strikes_greek]

plt.figure(figsize=(10, 5))
plt.plot(strikes_greek, gammas, 'purple', linewidth=2, label='Gamma')
plt.axvline(x=S, color='r', linestyle='--', alpha=0.5, label=f'Current Price ${S}')
plt.xlabel('Strike Price ($)')
plt.ylabel('Gamma')
plt.title('Option Gamma vs Strike Price')
plt.legend()
plt.grid(True)
plt.show()

### Vega: Sensitivity to Volatility

Vega shows how much the option price changes for a 1% change in volatility.

In [ ]:
# Vega visualization
vegas = [option_vega(S, K, T, r, sigma) for K in strikes_greek]

plt.figure(figsize=(10, 5))
plt.plot(strikes_greek, vegas, 'orange', linewidth=2, label='Vega')
plt.axvline(x=S, color='r', linestyle='--', alpha=0.5, label=f'Current Price ${S}')
plt.xlabel('Strike Price ($)')
plt.ylabel('Vega')
plt.title('Option Vega vs Strike Price')
plt.legend()
plt.grid(True)
plt.show()

## 5. Implied Volatility Calculation

Given a market option price, we can back out the implied volatility.

In [ ]:
# Implied Volatility Example
# First, calculate a theoretical option price
S = 100
K = 105
T = 0.5
r = 0.05
true_sigma = 0.25

theoretical_call_price = black_scholes_call(S, K, T, r, true_sigma)
print(f"Market Call Price: ${theoretical_call_price:.2f}")

# Now calculate implied volatility from this price
implied_vol = implied_volatility(theoretical_call_price, S, K, T, r, 'call')
print(f"True Volatility: {true_sigma*100:.2f}%")
print(f"Implied Volatility: {implied_vol*100:.2f}%")
print(f"Difference: {abs(true_sigma - implied_vol)*100:.4f}%")

### Implied Volatility Smile/Skew

Let's create a volatility smile by calculating IV across different strikes.

In [ ]:
# Volatility Smile
S = 100
T = 0.25
r = 0.05

# Create prices with a volatility smile pattern
strikes_iv = np.arange(85, 116, 2.5)
# Simulate a volatility smile (higher vol for OTM options)
true_vols = 0.20 + 0.15 * ((strikes_iv - S) / S) ** 2

call_market_prices = [black_scholes_call(S, K, T, r, vol) 
                      for K, vol in zip(strikes_iv, true_vols)]

# Calculate implied volatilities
implied_vols = []
for price, K in zip(call_market_prices, strikes_iv):
    try:
        iv = implied_volatility(price, S, K, T, r, 'call')
        implied_vols.append(iv)
    except:
        implied_vols.append(np.nan)

plt.figure(figsize=(10, 5))
plt.plot(strikes_iv, np.array(implied_vols) * 100, 'bo-', linewidth=2, markersize=6)
plt.axvline(x=S, color='r', linestyle='--', alpha=0.5, label=f'Spot Price ${S}')
plt.xlabel('Strike Price ($)')
plt.ylabel('Implied Volatility (%)')
plt.title('Volatility Smile')
plt.legend()
plt.grid(True)
plt.show()

## 6. Creating a Synthetic Option Chain

Generate a complete option chain for analysis.

In [ ]:
# Create Option Chain
S = 100
T = 0.5
r = 0.05
sigma = 0.22

option_chain = create_option_chain(
    S=S, 
    T=T, 
    r=r, 
    sigma=sigma, 
    strike_range=(0.8, 1.2),
    num_strikes=15
)

print("\nSynthetic Option Chain:")
print(f"Spot Price: ${S}")
print(f"Time to Expiration: {T} years")
print(f"Implied Volatility: {sigma*100}%\n")
print(option_chain.head(10))

In [ ]:
# Visualize Option Chain
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Call and Put prices
ax1.plot(option_chain['strike'], option_chain['call_price'], 
         'b-o', linewidth=2, markersize=5, label='Call Price')
ax1.plot(option_chain['strike'], option_chain['put_price'], 
         'g-s', linewidth=2, markersize=5, label='Put Price')
ax1.axvline(x=S, color='r', linestyle='--', alpha=0.5, label=f'Spot ${S}')
ax1.set_xlabel('Strike Price ($)')
ax1.set_ylabel('Option Price ($)')
ax1.set_title('Option Prices')
ax1.legend()
ax1.grid(True)

# Moneyness analysis
moneyness = option_chain['strike'] / S
ax2.scatter(moneyness, option_chain['call_price'], 
           c=option_chain['call_price'], cmap='Blues', s=100, label='Calls')
ax2.scatter(moneyness, option_chain['put_price'], 
           c=option_chain['put_price'], cmap='Greens', s=100, marker='s', label='Puts')
ax2.axvline(x=1.0, color='r', linestyle='--', alpha=0.5, label='ATM')
ax2.set_xlabel('Moneyness (K/S)')
ax2.set_ylabel('Option Price ($)')
ax2.set_title('Option Prices by Moneyness')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 7. Practical Example: Portfolio Hedging

Use options to hedge a stock position.

In [ ]:
# Hedging Example
# Assume we own 1000 shares of stock at $100
shares = 1000
S = 100
portfolio_value = shares * S

print(f"Portfolio: {shares} shares at ${S} = ${portfolio_value:,.2f}")

# Buy protective puts (10% OTM)
K_put = 90
T = 0.25  # 3 months
r = 0.05
sigma = 0.25

put_price = black_scholes_put(S, K_put, T, r, sigma)
total_hedge_cost = shares * put_price

print(f"\nHedge Strategy: Buy {shares} put options")
print(f"Strike Price: ${K_put}")
print(f"Put Price: ${put_price:.2f}")
print(f"Total Hedge Cost: ${total_hedge_cost:,.2f}")
print(f"Hedge Cost as % of Portfolio: {(total_hedge_cost/portfolio_value)*100:.2f}%")

# Calculate protected portfolio value at different stock prices
stock_prices = np.arange(70, 131, 1)
unhedged_values = shares * stock_prices
hedged_values = shares * np.maximum(stock_prices, K_put) - total_hedge_cost

plt.figure(figsize=(12, 6))
plt.plot(stock_prices, unhedged_values, 'b-', linewidth=2, label='Unhedged Portfolio')
plt.plot(stock_prices, hedged_values, 'g-', linewidth=2, label='Hedged Portfolio (Protected Put)')
plt.axvline(x=S, color='r', linestyle='--', alpha=0.5, label=f'Current Price ${S}')
plt.axhline(y=portfolio_value, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('Stock Price at Expiration ($)')
plt.ylabel('Portfolio Value ($)')
plt.title('Portfolio Hedging with Protective Puts')
plt.legend()
plt.grid(True)
plt.show()

## 8. Advanced: Volatility Surface Analysis

In [ ]:
# Volatility Surface
strikes_surf = np.arange(80, 121, 5)
times_surf = np.array([0.083, 0.25, 0.5, 1.0, 2.0])  # 1m, 3m, 6m, 1y, 2y

# Create a DataFrame for the surface
vol_surface_data = []

for T_val in times_surf:
    for K_val in strikes_surf:
        # Simulate a volatility surface (higher vol for longer maturities and OTM)
        base_vol = 0.20
        term_structure = 0.02 * np.sqrt(T_val)
        skew = 0.10 * ((K_val - S) / S) ** 2
        vol = base_vol + term_structure + skew
        
        vol_surface_data.append({
            'Strike': K_val,
            'Maturity': T_val,
            'Volatility': vol
        })

vol_surface_df = pd.DataFrame(vol_surface_data)
vol_pivot = vol_surface_df.pivot(index='Strike', columns='Maturity', values='Volatility')

print("\nVolatility Surface:")
print(vol_pivot * 100)  # Display as percentages

## Summary

This notebook demonstrated:
1. Pricing call and put options using Black-Scholes
2. Calculating and visualizing option Greeks
3. Computing implied volatility from market prices
4. Creating synthetic option chains
5. Practical hedging applications
6. Volatility surface analysis

### Key Takeaways:
- **Black-Scholes** provides theoretical prices for European options
- **Greeks** help understand risk exposures and sensitivities
- **Implied volatility** reveals market expectations
- **Options** can be used for hedging and risk management